# Find SeaTube organisms with Python

For researchers comfortable with basic Python. You will discover organism groups, inspect annotated taxa, filter observations, and plan frames and clips with provenance.

**The default walkthrough is entirely offline.** It uses six synthetic observations and bundled WoRMS lineages. The archive filename is fictitious. No ONC token, ffmpeg, or video downloads are needed. Install the package first with `python -m pip install -e ".[notebooks]"` from the repository root, then open this notebook in JupyterLab.

1. Inspect search definitions and the example dataset.
2. Compare broad and narrow biological queries.
3. Inspect coverage and plan media.
4. Optionally switch to real ONC data and explicitly enable extraction.

In [ ]:
from pathlib import Path
from seatube import AnnotationSet, OncClient, SeaTube, WormsResolver, ReviewFilters

RUN_LIVE = False          # Change only when ready to query ONC with your own .env token.
DOWNLOAD_MEDIA = False    # A separate opt-in; requires RUN_LIVE=True and ffmpeg.

if DOWNLOAD_MEDIA and not RUN_LIVE:
    raise ValueError("Synthetic example media does not exist. Use real data first.")

root = Path.cwd()
if not (root / "examples" / "data").exists():
    root = root.parent
fixture_dir = root / "examples" / "data"
assert fixture_dir.exists(), "Open the notebook from the repository root or examples directory."


## 1. Discover supported organisms

Groups are search rules, not guarantees of available observations. For example, `crabs` includes Brachyura and Anomura; `true-crabs` includes Brachyura only. The complete table is in [the organism catalog](../docs/organisms.md).

In [ ]:
from seatube import organism_groups

for group in organism_groups("crab"):
    print(group["group"], "→", ", ".join(group["ancestors"]))

## 2. Load a small dataset

By default these are **fabricated observations**, not an expedition dataset. The real-data branch is intentionally limited to the first dive in a small date range, so it is only an exploratory sample. A returned dive may have no relevant annotations. Remove or change the cap deliberately for a broader study.

In [ ]:
if RUN_LIVE:
    sea = SeaTube(data_dir=root / "downloads" / "tutorial")
    annotations = sea.fetch(
        "2019-07-06T00:00:00Z", "2019-07-06T23:59:59Z",
        max_dives=1,
        save_to=root / "downloads" / "tutorial" / "annotations.json",
    )
else:
    sea = SeaTube(
        client=OncClient(),
        resolver=WormsResolver(fixture_dir / "lineages.json", offline=True),
    )
    annotations = AnnotationSet.load(fixture_dir / "annotations.json")
    print("SYNTHETIC EXAMPLE — not observed ONC data")

annotations.summary()

## 3. What is actually annotated here?

Counts describe annotation records, not numbers of animals. Groups overlap; a crab also counts as a crustacean. `mapped_annotations` counts observations with a usable position inside a source archive. The synthetic fixture has six observations, five mapped to video, and four in the broad crab group.

In [ ]:
for row in sea.available_groups(annotations):
    print(f"{row['group']:16} {row['annotations']} annotated, {row['mapped_annotations']} mapped")

# Optional notebook table, if pandas is installed:
# import pandas as pd
# pd.DataFrame(sea.available_groups(annotations))

In [ ]:
for taxon in annotations.taxon_summary():
    print(taxon.name, taxon.aphia_id, taxon.annotations)

## 4. Narrow the biological question

The broad and narrow queries differ because Galatheoidea is an anomuran group. A list of organisms means **OR**. Additional depth, date and review conditions mean **AND**. Scientific-name searches can only recover identification detail actually recorded in the annotation or its lineage.

In [ ]:
crabs = sea.search(annotations, "crabs")
true_crabs = sea.search(annotations, "true-crabs")
print("Crabs:", len(crabs), "True crabs:", len(true_crabs))

reviewed = sea.search(
    annotations, "crabs",
    min_depth_m=500, max_depth_m=2000,
    review=ReviewFilters(reviewed_only=True),
)
print("Reviewed crab annotations in depth range:", len(reviewed))

if not RUN_LIVE:
    assert len(crabs) == 4 and len(true_crabs) == 3
    assert crabs.summary()["mapped_annotations"] == 3

### Exercise: find a sponge or a fish

Search for either organism in one query. Compare the annotation count with the number of frames you can plan. If you are using the fixture, both observations map to video. The following cell provides an answer scaffold.

In [ ]:
selected = sea.search(annotations, ["sponges", "fish"])
print(selected.summary())
print("Planned frames:", len(selected.frames()))
if not RUN_LIVE:
    assert len(selected) == 2

## 5. Plan a deliberately small download

Planning does not download video. `check_sizes=False` also avoids size requests. With live data, `describe_plan()` can query ONC for sizes. Unknown sizes remain unknown; `max_videos` caps source-file count, not bytes. The planner favors archives with more requested outputs, which is not an ecological sampling design.

In [ ]:
frames = crabs.frames(max_images=3, max_videos=1)
images = sea.image_downloader(root / "downloads" / "tutorial" / "images", keep_videos=True)
print(images.plan(frames, check_sizes=False))

clips = crabs.clips(before_seconds=1, after_seconds=2, max_clips=2, max_videos=1)
for clip in clips:
    print(clip)

if RUN_LIVE:
    print(images.describe_plan(frames))

## 6. Optional extraction from real ONC archives

Enable **both** switches at the top only after inspecting the live plan. The default cell prints a reminder and downloads nothing. A source archive is downloaded in full; frames and clips can share its cache. Clips merge overlapping intervals and stop at file boundaries. Organisms are not guaranteed visible throughout an excerpt.

In [ ]:
if RUN_LIVE and DOWNLOAD_MEDIA:
    image_rows = images.download(frames)
    video = sea.clip_downloader(
        root / "downloads" / "tutorial" / "clips",
        video_dir=root / "downloads" / "tutorial" / "images" / "_videos",
    )
    print(video.describe_plan(clips))
    clip_rows = video.download(clips)
    print(f"Indexed {len(image_rows)} images and {len(clip_rows)} clips")
else:
    print("No media downloaded. Set RUN_LIVE and DOWNLOAD_MEDIA to True to extract real data.")

## 7. Keep provenance and interpret zero matches carefully

Flat records are useful for pandas; CSV/JSONL exporters are available without pandas. Each media JSONL index includes original annotations. Archive filenames, time offsets, labels, annotators, and supplied geographic metadata travel with the output.

Missing matches can reflect the selected scope, generic labels, missing taxonomy, or recording gaps. `IncompleteTaxonomyWarning` means classification could not fully support the query; inspect `sea.resolver.unresolved`. Offline mode controls taxonomy only; do not call live discovery/fetch methods when working offline.

Extension: select observations by dive and split a machine-learning dataset by source dive or archive file, so adjacent frames do not leak across train and evaluation sets.

In [ ]:
flat_rows = crabs.flatten()
print("Flat annotation/taxon rows:", len(flat_rows))
# To save a study's subset:
# crabs.save("downloads/crab_annotations.json")
# crabs.write_flat_csv("downloads/crab_annotations.csv")
# crabs.write_flat_jsonl("downloads/crab_annotations.jsonl")
sea.close()